# Tutoriel de Données du Télescope Planck

**Tutoriel** : Ce tutoriel démontre comment charger des fichiers FITS et manipuler le format HEALPix pour visualiser des cartes du ciel et mener des analyses de base des données en Python.<br>
*Basé sur des observations obtenues avec Planck (http://www.esa.int/Planck), une mission scientifique de l'ESA avec des instruments et contributions directement financés par les États membres de l'ESA, la NASA, et le Canada.*<br>
**Mission et Instrument** : Télescope Planck <br>
**Cible Astronomique** : Analyse des anisotropies et polarisation du rayonnement cosmique de fond <br>
**Exigences du système** : Python 3.9+, MacOS/Linux ou WSL si sur Windows<br>
**Niveau du tutoriel** : Intermédiaire <br>

***
**Licence MIT** <br>
Copyright (c) Sa Majesté le Roi du chef du Canada, représentée par l'Agence spatiale canadienne, 2024. <br>
Droit d'auteur (c) Sa Majesté le Roi du chef du Canada, représentée par l'Agence Spatiale Canadienne, 2024.<br>

Pour plus d'informations, veuillez vous référer au fichier *License.txt*.

***
**Informations contextuelles** <br>
Cette mission de l'Agence spatiale européenne (ESA) a commencé le 14 mai 2009, avec le lancement d'une fusée Ariane 5 transportant l'Observatoire spatial Herschel, et s'est terminée en 2013. Les objectifs de la mission Planck étaient d'étudier les anisotropies et la polarisation du rayonnement cosmique de fond, ainsi que la naissance et l'évolution de l'Univers, et les formes qu'il pourrait prendre dans l'avenir. Le télescope spatial Planck transportait deux instruments : l'Instrument Haute Fréquence (HFI) et l'Instrument Basse Fréquence (LFI).

Pendant cette mission, l'ASC a financé l'analyse rapide et le logiciel de données de tendance, et a fourni un support scientifique pour les deux instruments, ainsi que la réduction de données et le support post-opérations jusqu'en 2016.

## Importations et Initialisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from astropy.io import fits
import healpy as hp

## Chargement des données

Les fichiers pour lesquels ce tutoriel est destiné peuvent être téléchargés depuis l'[Archive Héritée de Planck](https://pla.esac.esa.int/pla/#home). <br>
Vous pouvez les trouver sous *Maps -> Frequency maps -> Light maps*, avec chaque fréquence dans son propre fichier FITS. Téléchargez-les dans le même répertoire que ce notebook.<br>
Le code suivant a les noms des fichiers tels qu'ils sont actuellement au moment de la création de ce tutoriel (été 2024), décommentez simplement celui avec lequel vous souhaitez travailler.<br>
*Note : l'instrument HFI ne fournit que des cartes Q et U pour les canaux 100, 143, 217, et 353 GHz. Le code doit être modifié pour éviter que ces colonnes soient utilisées si vous chargez des cartes de fréquence plus élevée.*

In [ ]:
#filename = 'LFI_SkyMap_030-BPassCorrected-field-IQU_1024_R3.00_full.fits'
#filename = 'LFI_SkyMap_044-BPassCorrected-field-IQU_1024_R3.00_full.fits'
#filename = 'LFI_SkyMap_070-BPassCorrected-field-IQU_1024_R3.00_full.fits'
#filename = 'HFI_SkyMap_100-field-IQU_2048_R3.00_full.fits'
#filename = 'HFI_SkyMap_143-field-IQU_2048_R3.00_full.fits'
#filename = 'HFI_SkyMap_217-field-IQU_2048_R3.00_full.fits'
filename = 'HFI_SkyMap_353-psb-field-IQU_2048_R3.00_full.fits'
#filename = 'HFI_SkyMap_545-field-Int_2048_R3.00_full.fits' # Requires adjustments to the code.
#filename = 'HFI_SkyMap_857-field-Int_2048_R3.00_full.fits' # Requires adjustments to the code.

# Open the FITS file
with fits.open(filename) as hdul:
    
    # Print information about the file
    hdul.info()

    # Load the header
    header = hdul[1].header
    
    # Load the temperature data
    temperature_data = hdul[1].data

## Information d'en-tête
L'en-tête contient des informations sur la structure des données dans le fichier FITS. C'est un dictionnaire qui peut être indexé pour les valeurs contenues selon les besoins.

In [ ]:
print(header)

NSIDE = header["NSIDE"]
BAD_DATA = header["BAD_DATA"]

print("\nValeurs extraites:")
print(f'NSIDE: {NSIDE}') # NSIDE is the resolution parameter of the HEALPix projection
print(f'BAD_DATA: {BAD_DATA}') # This is the value assigned to erroneous pixels.

## Canaux de polarisation
Ces fichiers FITS contiennent généralement trois canaux de polarisation : <br>
<ul>
    <li> <b>I_STOKES</b> est la carte de l'intensité totale du rayonnement à chaque point du ciel, essentiellement une mesure de luminosité.<br></li>
    <li> <b>Q_STOKES</b> est une carte de la polarisation linéaire du rayonnement à chaque point le long de deux axes orthogonaux.<br></li>
    <li> <b>U_STOKES</b> est une carte de polarisation linéaire correspondante, mais avec des axes tournés de 45 degrés par rapport à <i>Q_STOKES</i>.<br></li>
</ul>
Pour chaque canal, nous extrayons la colonne du fichier FITS, puis la convertissons au format anneau, car elles sont stockées en format imbriqué. Healpy fonctionne en format anneau par défaut, donc cela économise quelques arguments supplémentaires dans les opérations subséquentes.

*Pour plus d'informations sur le format HEALPix, veuillez visiter [healpix.sourceforge.io](https://healpix.sourceforge.io/)*

In [ ]:
# Extract the temperature channel you wish to analyze. 
temperature_channel_I = temperature_data.field(0) # I_STOKES
temperature_channel_Q = temperature_data.field(1) # Q_STOKES (if working with the noted frequences, remove this and all references)
temperature_channel_U = temperature_data.field(2) # U_STOKES (same as above)

# Convert the HEALPix data from nested to ring format.
temperature_channel_I_ring = hp.pixelfunc.reorder(temperature_channel_I, n2r=True)
temperature_channel_Q_ring = hp.pixelfunc.reorder(temperature_channel_Q, n2r=True)
temperature_channel_U_ring = hp.pixelfunc.reorder(temperature_channel_U, n2r=True)

## Visualisation
Maintenant que nous avons extrait toutes les données du fichier FITS, nous pouvons y jeter un coup d'œil, en commençant par la carte de température *I_STOKES*.<br>
Puisqu'une carte du ciel est des données encodées dans une sphère 3D, nous devons utiliser une projection pour voir son intégralité en 2D. Un choix commun est la projection Mollweide :

In [ ]:
hp.mollview(temperature_channel_I_ring, title='Carte du Ciel des Données du Satellite Planck - Canal de Température I', unit='Kelvin (relatif au CMB)')
plt.show()

*Note : Les valeurs sont relatives à la température du rayonnement cosmique de fond (CMB), 2,725 Kelvin.*<br>
C'est un peu difficile à voir, essayons donc d'ajouter une normalisation par histogramme :

In [ ]:
hp.mollview(temperature_channel_I_ring, title='Carte du Ciel des Données du Satellite Planck - Canal de Température I, normalisé', unit='Kelvin (relatif au CMB)', norm='hist')
plt.show()

Ensuite, nous pouvons rendre le schéma de couleurs un peu plus familier en définissant une carte de couleurs personnalisée.
Celle-ci correspond à peu près à celle utilisée par l'ESA dans leurs visualisations :

In [ ]:
cdict = {
    'red':   ((0.0, 0.0, 0.0),
              (0.35, 0.0, 0.0),
              (0.5, 1.0, 1.0),
              (0.65, 1.0, 1.0),
              (1.0, 0.35, 0.35)),
    
    'green': ((0.0, 0.0, 0.0),
              (0.35, 1.0, 1.0),
              (0.5, 1.0, 1.0),
              (0.65, 0.5, 0.5),
              (1.0, 0.0, 0.0)),
    
    'blue':  ((0.0, 0.9, 0.9),
              (0.35, 1.0, 1.0),
              (0.5, 1.0, 1.0),
              (0.65, 0.0, 0.0),
              (1.0, 0.0, 0.0))
}

custom_cmap = LinearSegmentedColormap('CustomMap', cdict)


hp.mollview(temperature_channel_I_ring, title='Carte du Ciel des Données du Satellite Planck - Canal de Température I', unit='Kelvin (relatif au CMB)', norm='hist', cmap=custom_cmap)
plt.show()

Si vous souhaitez ajouter des lignes de grille au tracé, ajoutez simplement ce qui suit :

In [ ]:
hp.mollview(temperature_channel_I_ring, title='Carte du Ciel des Données du Satellite Planck - Canal de Température I', unit='Kelvin (relatif au CMB)', norm='hist', cmap=custom_cmap)
hp.graticule()

# Add galactic coordinate labels
for lon in range(0, 360, 30):
    hp.projtext(lon, 0, f'{lon}°', lonlat=True, color='black', ha='center', va='bottom')

for lat in range(-60, 90, 30):
    hp.projtext(0, lat, f'{lat}°', lonlat=True, color='black', ha='left', va='center')

plt.show()

Nous pouvons aussi jeter un coup d'œil de près à un endroit particulier.<br>
Le paramètre ***rot*** définit la coordonnée qui sera au centre. Une troisième valeur peut être ajoutée pour faire tourner la vue.<br>
Le paramètre ***reso*** définit la résolution angulaire/zoom de l'image. Plus la valeur est faible, plus la vue est zoomée. <br>
*Note : le sous-tracé résultant sera normalisé indépendamment, altérant les couleurs.*

In [ ]:
hp.gnomview(temperature_channel_I_ring, rot=[1, 10], title="Gros plan", unit='Kelvin (relatif au CMB)', norm='hist', cmap=custom_cmap, reso=2)
plt.show()

Jetons un coup d'œil à un des canaux de polarisation :

In [ ]:
hp.mollview(temperature_channel_Q_ring, title="Carte du Ciel des Données du Satellite Planck - Canal de Polarisation Q", cmap=custom_cmap, norm='hist', unit='Kelvin (relatif au CMB)')
plt.show()

Ces cartes sont beaucoup plus bruyantes que la première. <br>
Pour mieux les visualiser, nous pouvons utiliser la fonction de lissage fournie par Healpy pour les clarifier :

In [ ]:
smoothed_Q = hp.smoothing(temperature_channel_Q_ring, fwhm=np.radians(0.5))

hp.mollview(smoothed_Q, title="Carte de Polarisation Lissée - Q", cmap=custom_cmap, norm='hist', unit='Kelvin (relatif au CMB)')
plt.show()

## Analyse de Données

Maintenant que nous avons examiné visuellement les cartes du ciel, nous pouvons commencer à analyser les données elles-mêmes.
D'abord, traçons un histogramme des valeurs de pixels :

In [ ]:
plt.hist(temperature_channel_I_ring, bins=1000)
plt.title('Distribution de Température I_STOKES')
plt.xlabel('Température (Kelvin, relative au CMB)')
plt.ylabel('Nombre de Pixels')
plt.show()

Le graphique est un peu difficile à lire, mettons-le sur une échelle logarithmique pour mieux voir la distribution :

In [ ]:
plt.hist(temperature_channel_I_ring, bins=1000)
plt.title('Distribution de Température - I_STOKES (échelle logarithmique)') 
plt.yscale('log')
plt.xlabel('Température (Kelvin, relative au CMB)')
plt.ylabel('Nombre de Pixels')
plt.show()

Nous pouvons utiliser Healpy pour effectuer des transformées d'harmoniques sphériques, nous permettant de calculer le spectre de puissance angulaire de la carte :

In [ ]:
# Spherical Harmonics transforms
LMAX = 1024
cl = hp.anafast(temperature_channel_I_ring, lmax=LMAX)
ell = np.arange(len(cl))

# Plot the power spectrum
plt.figure(figsize=(10, 5))
plt.plot(ell, ell * (ell + 1) * cl)
plt.xlabel(r'$\ell$')
plt.ylabel(r'$\ell(\ell+1)C_{\ell}$')
plt.title('Spectre de Puissance Angulaire - I_STOKES')
plt.grid()
plt.show()